In [1]:
import pandas as pd
import numpy as np

ball = pd.read_csv("../data/ball_by_ball.csv")
matches = pd.read_csv("../data/matches.csv")
competitions = pd.read_csv("../data/competitions.csv")

In [2]:
t20_comp_ids = competitions.loc[
    competitions["match_format"] == "T20",
    "comp_id"
].unique()

t20_matches = matches[
    matches["comp_id"].isin(t20_comp_ids)
].copy()

t20_ball = ball[
    ball["match_id"].isin(t20_matches["match_id"])
].copy()

bowling_data = t20_ball.merge(
    t20_matches[["match_id", "comp_id", "match_date"]],
    on="match_id",
    how="left",
    validate="many_to_one"
)

bowling_data = bowling_data.merge(
    competitions[["comp_id", "comp_name", "season"]],
    on="comp_id",
    how="left",
    validate="many_to_one"
)

print("Bowling dataset shape:", bowling_data.shape)
print("Unique bowlers:", bowling_data["bowler_id"].nunique())

Bowling dataset shape: (94672, 37)
Unique bowlers: 352


In [3]:
print("EXTRAS TYPES")
display(bowling_data["extras_type"].value_counts(dropna=False))

print("\nDISMISSAL TYPES")
display(bowling_data["dismissal_type"].value_counts(dropna=False))

print("\nPACE OR SPIN")
display(bowling_data["pace_or_spin"].value_counts(dropna=False))

print("\nWICKET FLAG")
display(bowling_data["is_wicket"].value_counts(dropna=False))

print("\nBOWLING COLUMNS")
display(
    bowling_data[
        [
            "bowler_name",
            "runs",
            "is_wicket",
            "dismissal_type",
            "extras_type",
            "extras_2_runs",
            "pace_or_spin"
        ]
    ].head(15)
)

EXTRAS TYPES


extras_type
NaN         89077
wide         3792
legbyes      1196
Overstep      268
byes          212
other         127
Name: count, dtype: int64


DISMISSAL TYPES


dismissal_type
NaN                      89819
Caught                    3446
Bowled                     724
LBW                        298
RunOut                     275
Stump                       95
Hit Wicket                   7
Retire Out                   5
Obstracting the field        2
Obstructing the field        1
Name: count, dtype: int64


PACE OR SPIN


pace_or_spin
Pace    60365
Spin    34307
Name: count, dtype: int64


WICKET FLAG


is_wicket
False    89819
True      4853
Name: count, dtype: int64


BOWLING COLUMNS


,bowler_name,runs,is_wicket,dismissal_type,extras_type,extras_2_runs,pace_or_spin
0,Umesh Yadav,0,False,NaN,Overstep,0,Pace
1,Umesh Yadav,0,False,NaN,NaN,0,Pace
2,Umesh Yadav,0,False,NaN,wide,0,Pace
3,Umesh Yadav,0,False,NaN,NaN,0,Pace
4,Umesh Yadav,0,True,Caught,NaN,0,Pace
5,Umesh Yadav,0,False,NaN,wide,0,Pace
6,Umesh Yadav,0,False,NaN,NaN,0,Pace
7,Umesh Yadav,0,False,NaN,NaN,0,Pace
8,Umesh Yadav,0,False,NaN,NaN,0,Pace
9,Shivam Mavi,0,False,NaN,NaN,0,Pace


In [4]:
print("RUNS BY EXTRAS TYPE")

display(
    bowling_data.groupby("extras_type", dropna=False)
    .agg(
        deliveries=("runs", "count"),
        runs_total=("runs", "sum"),
        extras2_total=("extras_2_runs", "sum"),
        min_runs=("runs", "min"),
        max_runs=("runs", "max")
    )
)

print("\nEXTRAS FIELD COMBINATIONS")

display(
    bowling_data.groupby(
        ["extras_type", "extras1_runs", "extras_2_runs"],
        dropna=False
    ).size().reset_index(name="deliveries")
)

print("\nWICKET TYPES")

display(
    bowling_data[bowling_data["is_wicket"] == True]
    ["dismissal_type"].value_counts(dropna=False)
)

RUNS BY EXTRAS TYPE


,deliveries,runs_total,extras2_total,min_runs,max_runs
extras_type,,,,,
Overstep,268,398,28,0,6
byes,212,0,0,0,0
legbyes,1196,1,0,0,1
other,127,210,20,0,6
wide,3792,0,0,0,0
NaN,89077,131419,0,0,6



EXTRAS FIELD COMBINATIONS


,extras_type,extras1_runs,extras_2_runs,deliveries
0,Overstep,1,0,255
1,Overstep,1,1,8
2,Overstep,1,4,5
3,byes,1,0,139
4,byes,2,0,16
5,byes,3,0,2
6,byes,4,0,55
7,legbyes,0,0,1
8,legbyes,1,0,1034
9,legbyes,2,0,52



WICKET TYPES


dismissal_type
Caught                   3446
Bowled                    724
LBW                       298
RunOut                    275
Stump                      95
Hit Wicket                  7
Retire Out                  5
Obstracting the field       2
Obstructing the field       1
Name: count, dtype: int64

In [5]:
# Inspect records where extras_2_runs is nonzero

extra_records = bowling_data[
    bowling_data["extras_2_runs"] > 0
]

display(
    extra_records[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "is_four",
            "is_six",
            "extras_type",
            "extras1_runs",
            "extras_2_runs"
        ]
    ].head(30)
)

print("Total records with extras_2_runs > 0:",
      len(extra_records))

,match_id,innings,overid,over_ball_num,innings_ball_num,striker_name,bowler_name,runs,is_four,is_six,extras_type,extras1_runs,extras_2_runs
4416,14112,1,11,5,66,David Warner,Varun Chakravarthy,0,False,False,Overstep,1,4
8195,14127,2,9,3,52,Prithvi Shaw,Obed McCoy,0,False,False,other,1,4
15653,14159,2,9,1,51,Ishan Kishan,Umran Malik,0,False,False,other,1,1
19358,20186,2,20,2,130,Krishnappa Gowtham,Tushar Deshpande,0,False,False,Overstep,1,1
19721,20188,1,19,4,117,M Shahrukh Khan,KM Asif,0,False,False,other,1,1
22471,20199,2,10,4,59,Nitish Rana,Marco Jansen,0,False,False,Overstep,1,1
22853,20201,1,11,5,66,Krunal Pandya,Sikandar Raza,0,False,False,Overstep,1,4
27059,20218,1,16,5,101,Marcus Stoinis,Kagiso Rabada,0,False,False,Overstep,1,1
30929,20234,2,13,3,80,Nehal Wadhera,Harshal Patel,0,False,False,other,1,4
33081,20243,2,19,4,121,Tim David,Naveen ul Haq,0,False,False,other,1,4


Total records with extras_2_runs > 0: 21


In [6]:
overstep_records = bowling_data[
    bowling_data["extras_type"] == "Overstep"
]

display(
    overstep_records[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "innings_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "extras_type",
            "extras_2_runs"
        ]
    ].head(30)
)

,match_id,innings,overid,over_ball_num,innings_ball_num,striker_name,bowler_name,runs,extras_type,extras_2_runs
0,14094,1,1,1,1,Ruturaj Gaikwad,Umesh Yadav,0,Overstep,0
978,14098,1,1,5,5,Jos Buttler,Bhuvneshwar Kumar,0,Overstep,0
990,14098,1,3,4,17,Yashasvi Jaiswal,Bhuvneshwar Kumar,0,Overstep,0
996,14098,1,4,3,23,Jos Buttler,Umran Malik,4,Overstep,0
1001,14098,1,5,1,28,Yashasvi Jaiswal,Washington Sundar,1,Overstep,0
1261,14099,1,6,3,33,Nitish Rana,Akash Deep,0,Overstep,0
1419,14099,2,13,2,76,Shahbaz Ahmed,Andre Russell,1,Overstep,0
1770,14101,1,9,3,54,Liam Livingstone,Umesh Yadav,0,Overstep,0
1908,14101,2,12,6,74,Andre Russell,Odean Smith,1,Overstep,0
1920,14101,2,14,4,86,Andre Russell,Arshdeep Singh,4,Overstep,0


In [7]:
unusual_records = bowling_data[
    bowling_data["extras_2_runs"] > 0
]

display(
    unusual_records[
        [
            "match_id",
            "innings",
            "overid",
            "over_ball_num",
            "striker_name",
            "bowler_name",
            "runs",
            "is_four",
            "is_six",
            "is_wicket",
            "dismissal_type",
            "extras_type",
            "extras1_runs",
            "extras_2_runs"
        ]
    ].to_string(index=False)
)

' match_id  innings  overid  over_ball_num        striker_name         bowler_name  runs  is_four  is_six  is_wicket dismissal_type extras_type  extras1_runs  extras_2_runs\n    14112        1      11              5        David Warner  Varun Chakravarthy     0    False   False      False            NaN    Overstep             1              4\n    14127        2       9              3        Prithvi Shaw          Obed McCoy     0    False   False      False            NaN       other             1              4\n    14159        2       9              1        Ishan Kishan         Umran Malik     0    False   False      False            NaN       other             1              1\n    20186        2      20              2  Krishnappa Gowtham    Tushar Deshpande     0    False   False      False            NaN    Overstep             1              1\n    20188        1      19              4     M Shahrukh Khan             KM Asif     0    False   False      False            NaN    

In [8]:
print("EXTRAS TYPE COUNTS WITH NONZERO EXTRAS_2_RUNS")

display(
    unusual_records.groupby("extras_type")
    .agg(
        records=("match_id", "count"),
        total_runs=("runs", "sum"),
        extra2_total=("extras_2_runs", "sum")
    )
)

EXTRAS TYPE COUNTS WITH NONZERO EXTRAS_2_RUNS


,records,total_runs,extra2_total
extras_type,,,
Overstep,13,0,28
other,8,0,20


In [9]:
# Check all unique values and combinations
print("extras1_runs:")
display(bowling_data["extras1_runs"].value_counts(dropna=False))

print("\nextras_2_runs:")
display(bowling_data["extras_2_runs"].value_counts(dropna=False))

print("\nNon-zero extras_2_runs by extras type:")
display(
    bowling_data[
        bowling_data["extras_2_runs"] > 0
    ].groupby(
        ["extras_type", "extras_2_runs"]
    ).size().unstack(fill_value=0)
)

extras1_runs:


extras1_runs
0    89078
1     5086
2      199
4      163
5      132
3       14
Name: count, dtype: int64


extras_2_runs:


extras_2_runs
0    94651
1       12
4        9
Name: count, dtype: int64


Non-zero extras_2_runs by extras type:


extras_2_runs,1,4
extras_type,,
Overstep,8,5
other,4,4


In [10]:
# Calculate provisional runs conceded per delivery

bowling_data["provisional_runs_conceded"] = (
    bowling_data["runs"] +
    bowling_data["extras_2_runs"].fillna(0)
)

# Inspect totals by extra type
display(
    bowling_data.groupby("extras_type", dropna=False)
    .agg(
        deliveries=("runs", "count"),
        batter_runs=("runs", "sum"),
        additional_extras=("extras_2_runs", "sum"),
        provisional_runs=("provisional_runs_conceded", "sum")
    )
)

,deliveries,batter_runs,additional_extras,provisional_runs
extras_type,,,,
Overstep,268,398,28,426
byes,212,0,0,0
legbyes,1196,1,0,1
other,127,210,20,230
wide,3792,0,0,0
NaN,89077,131419,0,131419


In [11]:
bowling_data["bowler_runs_conceded"] = (
    bowling_data["runs"] +
    bowling_data["extras_2_runs"].fillna(0)
)

# Add the standard one-run penalty for wides and no-balls
penalty_deliveries = ["wide", "Overstep"]

bowling_data.loc[
    bowling_data["extras_type"].isin(penalty_deliveries),
    "bowler_runs_conceded"
] += 1

display(
    bowling_data.groupby("extras_type", dropna=False)
    .agg(
        deliveries=("runs", "count"),
        batter_runs=("runs", "sum"),
        additional_extras=("extras_2_runs", "sum"),
        bowler_runs_conceded=("bowler_runs_conceded", "sum")
    )
)

,deliveries,batter_runs,additional_extras,bowler_runs_conceded
extras_type,,,,
Overstep,268,398,28,694
byes,212,0,0,0
legbyes,1196,1,0,1
other,127,210,20,230
wide,3792,0,0,3792
NaN,89077,131419,0,131419


In [12]:
# Wicket types credited to the bowler
bowler_wicket_types = [
    "Caught",
    "Bowled",
    "LBW",
    "Stump",
    "Hit Wicket"
]

bowling_data["bowler_wicket"] = (
    (bowling_data["is_wicket"] == True) &
    (bowling_data["dismissal_type"].isin(bowler_wicket_types))
)

print("Total dismissals:", bowling_data["is_wicket"].sum())
print("Wickets credited to bowlers:", bowling_data["bowler_wicket"].sum())

print("\nWickets by dismissal type:")
display(
    bowling_data[bowling_data["is_wicket"] == True]
    ["dismissal_type"]
    .value_counts(dropna=False)
)

Total dismissals: 4853
Wickets credited to bowlers: 4570

Wickets by dismissal type:


dismissal_type
Caught                   3446
Bowled                    724
LBW                       298
RunOut                    275
Stump                      95
Hit Wicket                  7
Retire Out                  5
Obstracting the field       2
Obstructing the field       1
Name: count, dtype: int64

In [13]:
print(bowling_data.columns.tolist())

['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time', 'comp_id', 'match_date', 'comp_name', 'season', 'provisional_runs_conceded', 'bowler_runs_conceded', 'bowler_wicket']


In [14]:
# Legal delivery: wides and no-balls are not legal balls
bowling_data["legal_ball"] = ~bowling_data["extras_type"].isin(
    ["wide", "Overstep"]
)

# Wickets credited to the bowler
bowler_wicket_types = [
    "Caught", "Bowled", "LBW", "Stump", "Hit Wicket"
]

bowling_data["bowler_wicket"] = (
    (bowling_data["is_wicket"] == True) &
    (bowling_data["dismissal_type"].isin(bowler_wicket_types))
)

# Verify
print(
    bowling_data[
        ["legal_ball", "bowler_wicket", "bowler_runs_conceded"]
    ].head()
)

   legal_ball  bowler_wicket  bowler_runs_conceded
0       False          False                     1
1        True          False                     0
2       False          False                     1
3        True          False                     0
4        True           True                     0


In [15]:
# Start with batter runs
bowling_data["bowler_runs_conceded"] = bowling_data["runs"].copy()

# Byes and leg-byes are not charged to the bowler
bowling_data.loc[
    bowling_data["extras_type"].isin(["byes", "legbyes"]),
    "bowler_runs_conceded"
] = 0

# Add the standard one-run penalty for wides and no-balls
bowling_data.loc[
    bowling_data["extras_type"].isin(["wide", "Overstep"]),
    "bowler_runs_conceded"
] += 1

# Add additional no-ball extras
bowling_data.loc[
    bowling_data["extras_type"] == "Overstep",
    "bowler_runs_conceded"
] += bowling_data.loc[
    bowling_data["extras_type"] == "Overstep",
    "extras_2_runs"
].fillna(0)

bowling_stats = (
    bowling_data.groupby(["bowler_id", "bowler_name"])
    .agg(
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("legal_ball", "sum"),
        wickets=("bowler_wicket", "sum"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

# Calculate overs and economy rate
bowling_stats["overs"] = bowling_stats["legal_balls"] / 6

bowling_stats["economy"] = (
    bowling_stats["runs_conceded"] /
    bowling_stats["overs"].replace(0, np.nan)
)

# Calculate bowling average and strike rate
bowling_stats["bowling_average"] = (
    bowling_stats["runs_conceded"] /
    bowling_stats["wickets"].replace(0, np.nan)
)

bowling_stats["bowling_strike_rate"] = (
    bowling_stats["legal_balls"] /
    bowling_stats["wickets"].replace(0, np.nan)
)

bowling_stats = bowling_stats.round(2)

display(
    bowling_stats.sort_values(
        ["wickets", "economy"],
        ascending=[False, True]
    ).head(20)
)

,bowler_id,bowler_name,runs_conceded,legal_balls,wickets,matches_played,overs,economy,bowling_average,bowling_strike_rate
34,1612,Yuzvendra Chahal,2314,1589,94,71,264.83,8.74,24.62,16.90
124,4722,Rashid Khan,2358,1718,86,77,286.33,8.24,27.42,19.98
9,360,Bhuvneshwar Kumar,2312,1651,84,74,275.17,8.40,27.52,19.65
171,6464,Arshdeep Singh,2429,1576,81,72,262.67,9.25,29.99,19.46
152,5944,Mohammed Siraj,2447,1647,78,75,274.50,8.91,31.37,21.12
157,6256,Varun Chakravarthy,1909,1396,75,63,232.67,8.20,25.45,18.61
42,2261,Harshal Patel,1892,1251,73,60,208.50,9.07,25.92,17.14
90,3644,Kuldeep Yadav,1922,1390,72,64,231.67,8.30,26.69,19.31
87,3628,Kagiso Rabada,1792,1149,72,51,191.50,9.36,24.89,15.96
45,2320,Trent Boult,1934,1363,69,62,227.17,8.51,28.03,19.75


In [16]:
# Keep IPL competitions only
ipl_bowling_data = bowling_data[
    bowling_data["comp_name"].str.startswith("IPL", na=False)
].copy()

print("IPL bowling dataset shape:", ipl_bowling_data.shape)
print("Unique IPL bowlers:", ipl_bowling_data["bowler_id"].nunique())
print("IPL matches:", ipl_bowling_data["match_id"].nunique())

print("\nIPL seasons:")
print(sorted(ipl_bowling_data["season"].dropna().unique()))

IPL bowling dataset shape: (87687, 41)
Unique IPL bowlers: 260
IPL matches: 367

IPL seasons:
['2022', '2023', '2024', '2025', '2026']


In [17]:
ipl_bowling_stats = (
    ipl_bowling_data.groupby(["bowler_id", "bowler_name"])
    .agg(
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("legal_ball", "sum"),
        wickets=("bowler_wicket", "sum"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

ipl_bowling_stats["overs"] = ipl_bowling_stats["legal_balls"] / 6

ipl_bowling_stats["economy"] = (
    ipl_bowling_stats["runs_conceded"] /
    ipl_bowling_stats["overs"].replace(0, np.nan)
)

ipl_bowling_stats["bowling_average"] = (
    ipl_bowling_stats["runs_conceded"] /
    ipl_bowling_stats["wickets"].replace(0, np.nan)
)

ipl_bowling_stats["bowling_strike_rate"] = (
    ipl_bowling_stats["legal_balls"] /
    ipl_bowling_stats["wickets"].replace(0, np.nan)
)

ipl_bowling_stats = ipl_bowling_stats.round(2)

display(
    ipl_bowling_stats.sort_values(
        ["wickets", "economy"],
        ascending=[False, True]
    ).head(20)
)

,bowler_id,bowler_name,runs_conceded,legal_balls,wickets,matches_played,overs,economy,bowling_average,bowling_strike_rate
34,1612,Yuzvendra Chahal,2314,1589,94,71,264.83,8.74,24.62,16.90
112,4722,Rashid Khan,2358,1718,86,77,286.33,8.24,27.42,19.98
9,360,Bhuvneshwar Kumar,2312,1651,84,74,275.17,8.40,27.52,19.65
154,6464,Arshdeep Singh,2429,1576,81,72,262.67,9.25,29.99,19.46
137,5944,Mohammed Siraj,2447,1647,78,75,274.50,8.91,31.37,21.12
141,6256,Varun Chakravarthy,1909,1396,75,63,232.67,8.20,25.45,18.61
42,2261,Harshal Patel,1892,1251,73,60,208.50,9.07,25.92,17.14
84,3644,Kuldeep Yadav,1922,1390,72,64,231.67,8.30,26.69,19.31
81,3628,Kagiso Rabada,1792,1149,72,51,191.50,9.36,24.89,15.96
45,2320,Trent Boult,1934,1363,69,62,227.17,8.51,28.03,19.75


In [18]:
print("IPL dataset shape:", ipl_bowling_data.shape)

print("\nCompetition names included:")
print(ipl_bowling_data["comp_name"].value_counts())

print("\nUnique IPL matches:",
      ipl_bowling_data["match_id"].nunique())

print("\nSeasons:")
print(sorted(ipl_bowling_data["season"].dropna().unique()))

print("\nAll T20 deliveries:", len(bowling_data))
print("IPL deliveries:", len(ipl_bowling_data))

IPL dataset shape: (87687, 41)

Competition names included:
comp_name
IPL 2022    17909
IPL 2023    17863
IPL 2026    17527
IPL 2025    17285
IPL 2024    17103
Name: count, dtype: int64

Unique IPL matches: 367

Seasons:
['2022', '2023', '2024', '2025', '2026']

All T20 deliveries: 94672
IPL deliveries: 87687


In [19]:

# Filter recent IPL seasons
recent_bowling_data = ipl_bowling_data[
    ipl_bowling_data["season"].isin(["2025", "2026"])
].copy()

print("Recent IPL deliveries:", len(recent_bowling_data))
print("Recent IPL matches:", recent_bowling_data["match_id"].nunique())
print("Recent IPL bowlers:", recent_bowling_data["bowler_id"].nunique())

Recent IPL deliveries: 34812
Recent IPL matches: 148
Recent IPL bowlers: 170


In [21]:
recent_bowling_stats = (
    recent_bowling_data.groupby(["bowler_id", "bowler_name"])
    .agg(
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("legal_ball", "sum"),
        wickets=("bowler_wicket", "sum"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

recent_bowling_stats["overs"] = (
    recent_bowling_stats["legal_balls"] / 6
)

recent_bowling_stats["economy"] = (
    recent_bowling_stats["runs_conceded"] /
    recent_bowling_stats["overs"].replace(0, np.nan)
)

recent_bowling_stats["bowling_average"] = (
    recent_bowling_stats["runs_conceded"] /
    recent_bowling_stats["wickets"].replace(0, np.nan)
)

recent_bowling_stats["bowling_strike_rate"] = (
    recent_bowling_stats["legal_balls"] /
    recent_bowling_stats["wickets"].replace(0, np.nan)
)

recent_bowling_stats = recent_bowling_stats.round(2)

display(
    recent_bowling_stats.sort_values(
        ["wickets", "economy"],
        ascending=[False, True]
    ).head(25)
)

,bowler_id,bowler_name,runs_conceded,legal_balls,wickets,matches_played,overs,economy,bowling_average,bowling_strike_rate
3,360,Bhuvneshwar Kumar,983,690,45,30,115.00,8.55,21.84,15.33
60,4489,Prasidh Krishna,894,588,41,27,98.00,9.12,21.80,14.34
122,7967,Noor Ahmad,825,606,37,28,101.00,8.17,22.30,16.38
7,830,Josh Hazlewood,880,573,37,25,95.50,9.21,23.78,15.49
37,3298,Jofra Archer,973,634,36,28,105.67,9.21,27.03,17.61
77,5944,Mohammed Siraj,1080,714,35,32,119.00,9.08,30.86,20.40
92,6464,Arshdeep Singh,1050,669,35,30,111.50,9.42,30.00,19.11
157,39657,Eshan Malinga,745,486,33,22,81.00,9.20,22.58,14.73
51,4219,Krunal Pandya,802,579,31,30,96.50,8.31,25.87,18.68
40,3628,Kagiso Rabada,787,472,31,21,78.67,10.00,25.39,15.23


In [22]:
# Season-wise IPL bowling statistics
season_bowling_stats = (
    recent_bowling_data.groupby(
        ["bowler_id", "bowler_name", "season"]
    )
    .agg(
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("legal_ball", "sum"),
        wickets=("bowler_wicket", "sum"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

season_bowling_stats["overs"] = (
    season_bowling_stats["legal_balls"] / 6
)

season_bowling_stats["economy"] = (
    season_bowling_stats["runs_conceded"] /
    season_bowling_stats["overs"].replace(0, np.nan)
)

season_bowling_stats["bowling_average"] = (
    season_bowling_stats["runs_conceded"] /
    season_bowling_stats["wickets"].replace(0, np.nan)
)

season_bowling_stats["bowling_strike_rate"] = (
    season_bowling_stats["legal_balls"] /
    season_bowling_stats["wickets"].replace(0, np.nan)
)

season_bowling_stats = season_bowling_stats.round(2)

display(
    season_bowling_stats.sort_values(
        ["season", "wickets"],
        ascending=[True, False]
    ).head(40)
)

,bowler_id,bowler_name,season,runs_conceded,legal_balls,wickets,matches_played,overs,economy,bowling_average,bowling_strike_rate
93,4489,Prasidh Krishna,2025,487,354,25,15,59.00,8.25,19.48,14.16
190,7967,Noor Ahmad,2025,400,300,24,14,50.00,8.00,16.67,12.50
11,830,Josh Hazlewood,2025,386,264,22,12,44.00,8.77,17.55,12.00
31,2320,Trent Boult,2025,513,346,22,16,57.67,8.90,23.32,15.73
147,6464,Arshdeep Singh,2025,513,350,21,16,58.33,8.79,24.43,16.67
114,5577,Sai Kishore,2025,393,255,19,15,42.50,9.25,20.68,13.42
54,2933,Jasprit Bumrah,2025,316,284,18,12,47.33,6.68,17.56,15.78
4,360,Bhuvneshwar Kumar,2025,483,312,17,14,52.00,9.29,28.41,18.35
80,4219,Krunal Pandya,2025,377,276,17,15,46.00,8.20,22.18,16.24
128,6256,Varun Chakravarthy,2025,383,300,17,13,50.00,7.66,22.53,17.65


In [23]:
# 2026 IPL bowling performance

display(
    season_bowling_stats[
        season_bowling_stats["season"] == "2026"
    ]
    .sort_values(
        ["wickets", "economy"],
        ascending=[False, True]
    )
    .head(20)
)

,bowler_id,bowler_name,season,runs_conceded,legal_balls,wickets,matches_played,overs,economy,bowling_average,bowling_strike_rate
64,3628,Kagiso Rabada,2026,625,388,29,17,64.67,9.66,21.55,13.38
5,360,Bhuvneshwar Kumar,2026,500,378,28,16,63.00,7.94,17.86,13.50
58,3298,Jofra Archer,2026,547,360,25,16,60.00,9.12,21.88,14.40
100,4722,Rashid Khan,2026,508,341,21,17,56.83,8.94,24.19,16.24
222,27964,Anshul Kamboj,2026,530,302,21,14,50.33,10.53,25.24,14.38
236,39657,Eshan Malinga,2026,507,326,20,15,54.33,9.33,25.35,16.30
121,5944,Mohammed Siraj,2026,558,372,19,17,62.00,9.00,29.37,19.58
173,7327,Rasikh Salam,2026,405,257,19,12,42.83,9.46,21.32,13.53
197,8530,Kartik Tyagi,2026,494,310,18,13,51.67,9.56,27.44,17.22
8,781,Sunil Narine,2026,340,309,17,13,51.50,6.60,20.00,18.18


In [24]:
# Compare 2025 and 2026 bowling performance

bowling_comparison = season_bowling_stats.pivot(
    index=["bowler_id", "bowler_name"],
    columns="season",
    values=[
        "wickets",
        "economy",
        "bowling_average",
        "bowling_strike_rate",
        "matches_played"
    ]
)

bowling_comparison.columns = [
    f"{metric}_{season}"
    for metric, season in bowling_comparison.columns
]

bowling_comparison = bowling_comparison.reset_index()

# Keep players who appeared in both seasons
bowling_comparison = bowling_comparison.dropna(
    subset=["wickets_2025", "wickets_2026"]
)

bowling_comparison["wicket_change"] = (
    bowling_comparison["wickets_2026"] -
    bowling_comparison["wickets_2025"]
)

bowling_comparison["economy_change"] = (
    bowling_comparison["economy_2026"] -
    bowling_comparison["economy_2025"]
)

bowling_comparison = bowling_comparison.round(2)

display(
    bowling_comparison.sort_values(
        ["wickets_2026", "economy_2026"],
        ascending=[False, True]
    ).head(30)
)

,bowler_id,bowler_name,wickets_2025,wickets_2026,economy_2025,economy_2026,bowling_average_2025,bowling_average_2026,bowling_strike_rate_2025,bowling_strike_rate_2026,matches_played_2025,matches_played_2026,wicket_change,economy_change
40,3628,Kagiso Rabada,2.0,29.0,11.57,9.66,81.00,21.55,42.00,13.38,4.0,17.0,27.0,-1.91
3,360,Bhuvneshwar Kumar,17.0,28.0,9.29,7.94,28.41,17.86,18.35,13.50,14.0,16.0,11.0,-1.35
37,3298,Jofra Archer,11.0,25.0,9.33,9.12,38.73,21.88,24.91,14.40,12.0,16.0,14.0,-0.21
63,4722,Rashid Khan,9.0,21.0,9.35,8.94,57.11,24.19,36.67,16.24,15.0,17.0,12.0,-0.41
145,27964,Anshul Kamboj,8.0,21.0,8.00,10.53,21.50,25.24,16.12,14.38,8.0,14.0,13.0,2.53
157,39657,Eshan Malinga,13.0,20.0,8.92,9.33,18.31,25.35,12.31,16.30,7.0,15.0,7.0,0.41
77,5944,Mohammed Siraj,16.0,19.0,9.16,9.00,32.62,29.37,21.38,19.58,15.0,17.0,3.0,-0.16
110,7327,Rasikh Salam,1.0,19.0,11.67,9.46,70.00,21.32,36.00,13.53,2.0,12.0,18.0,-2.21
5,781,Sunil Narine,12.0,17.0,7.80,6.60,29.25,20.00,22.50,18.18,12.0,13.0,5.0,-1.20
167,58246,Prince Yadav,3.0,16.0,9.85,8.86,75.00,28.88,45.67,19.56,6.0,14.0,13.0,-0.99


In [25]:
# Check over numbering

print("Minimum overid:", ipl_bowling_data["overid"].min())
print("Maximum overid:", ipl_bowling_data["overid"].max())

print("\nOver ID values:")
print(sorted(ipl_bowling_data["overid"].dropna().unique()))

Minimum overid: 1
Maximum overid: 20

Over ID values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20)]


In [26]:
# Assign bowling phases
phase_data = recent_bowling_data.copy()

phase_data["phase"] = np.select(
    [
        phase_data["overid"].between(1, 6),
        phase_data["overid"].between(7, 15),
        phase_data["overid"].between(16, 20)
    ],
    [
        "Powerplay",
        "Middle",
        "Death"
    ],
    default="Unknown"
)

# Remove any records outside the three phases
phase_data = phase_data[
    phase_data["phase"] != "Unknown"
].copy()

# Calculate phase-wise statistics
phase_bowling_stats = (
    phase_data.groupby(
        ["bowler_id", "bowler_name", "phase"]
    )
    .agg(
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("legal_ball", "sum"),
        wickets=("bowler_wicket", "sum"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

phase_bowling_stats["overs"] = (
    phase_bowling_stats["legal_balls"] / 6
)

phase_bowling_stats["economy"] = (
    phase_bowling_stats["runs_conceded"] /
    phase_bowling_stats["overs"].replace(0, np.nan)
)

phase_bowling_stats["bowling_average"] = (
    phase_bowling_stats["runs_conceded"] /
    phase_bowling_stats["wickets"].replace(0, np.nan)
)

phase_bowling_stats["bowling_strike_rate"] = (
    phase_bowling_stats["legal_balls"] /
    phase_bowling_stats["wickets"].replace(0, np.nan)
)

phase_bowling_stats = phase_bowling_stats.round(2)

# Avoid ranking players with very small samples:
# require at least 60 legal balls (10 overs) in that phase
phase_bowling_stats = phase_bowling_stats[
    phase_bowling_stats["legal_balls"] >= 60
]

# Display the top 15 wicket-takers in each phase
for phase in ["Powerplay", "Middle", "Death"]:
    print(f"\n{'=' * 20} {phase.upper()} {'=' * 20}")

    display(
        phase_bowling_stats[
            phase_bowling_stats["phase"] == phase
        ]
        .sort_values(
            ["wickets", "economy"],
            ascending=[False, True]
        )
        .head(15)
    )


==================== POWERPLAY ====================


,bowler_id,bowler_name,phase,runs_conceded,legal_balls,wickets,matches_played,overs,economy,bowling_average,bowling_strike_rate
11,360,Bhuvneshwar Kumar,Powerplay,484,390,25,30,65.00,7.45,19.36,15.60
197,5944,Mohammed Siraj,Powerplay,709,522,24,32,87.00,8.15,29.54,21.75
109,3628,Kagiso Rabada,Powerplay,530,312,21,21,52.00,10.19,25.24,14.86
100,3298,Jofra Archer,Powerplay,527,354,20,28,59.00,8.93,26.35,17.70
22,830,Josh Hazlewood,Powerplay,420,288,17,25,48.00,8.75,24.71,16.94
234,6464,Arshdeep Singh,Powerplay,531,360,15,30,60.00,8.85,35.40,24.00
37,1566,Mohammed Shami,Powerplay,451,313,14,22,52.17,8.65,32.21,22.36
48,1813,Deepak Chahar,Powerplay,436,300,14,22,50.00,8.72,31.14,21.43
170,4816,Khaleel Ahmed,Powerplay,429,300,12,19,50.00,8.58,35.75,25.00
369,27964,Anshul Kamboj,Powerplay,387,252,12,22,42.00,9.21,32.25,21.00



==================== MIDDLE ====================


,bowler_id,bowler_name,phase,runs_conceded,legal_balls,wickets,matches_played,overs,economy,bowling_average,bowling_strike_rate
307,7967,Noor Ahmad,Middle,640,480,28,28,80.00,8.00,22.86,17.14
164,4722,Rashid Khan,Middle,767,527,25,32,87.83,8.73,30.68,21.08
132,4219,Krunal Pandya,Middle,649,492,24,29,82.00,7.91,27.04,20.50
155,4489,Prasidh Krishna,Middle,443,330,20,26,55.00,8.05,22.15,16.50
116,3644,Kuldeep Yadav,Middle,628,444,20,25,74.00,8.49,31.40,22.20
41,1612,Yuzvendra Chahal,Middle,695,450,20,24,75.00,9.27,34.75,22.50
205,6256,Varun Chakravarthy,Middle,522,402,19,24,67.00,7.79,27.47,21.16
4,152,Ravindra Jadeja,Middle,567,403,19,26,67.17,8.44,29.84,21.21
186,5577,Sai Kishore,Middle,348,222,15,15,37.00,9.41,23.20,14.80
15,781,Sunil Narine,Middle,518,420,14,25,70.00,7.40,37.00,30.00



==================== DEATH ====================


,bowler_id,bowler_name,phase,runs_conceded,legal_balls,wickets,matches_played,overs,economy,bowling_average,bowling_strike_rate
9,360,Bhuvneshwar Kumar,Death,415,252,18,25,42.00,9.88,23.06,14.00
154,4489,Prasidh Krishna,Death,303,168,18,21,28.00,10.82,16.83,9.33
394,39657,Eshan Malinga,Death,260,156,15,19,26.00,10.00,17.33,10.40
232,6464,Arshdeep Singh,Death,336,195,15,25,32.50,10.34,22.40,13.00
17,829,Mitchell Starc,Death,146,90,14,14,15.00,9.73,10.43,6.43
333,8746,Vaibhav Arora,Death,266,153,14,19,25.50,10.43,19.00,10.93
367,27964,Anshul Kamboj,Death,238,131,14,13,21.83,10.90,17.00,9.36
54,2320,Trent Boult,Death,185,119,12,20,19.83,9.33,15.42,9.92
20,830,Josh Hazlewood,Death,283,164,12,20,27.33,10.35,23.58,13.67
299,7808,Marco Jansen,Death,250,135,12,20,22.50,11.11,20.83,11.25


In [27]:
# Check how runs are currently charged to bowlers

display(
    bowling_data.groupby("extras_type", dropna=False)
    .agg(
        deliveries=("runs", "size"),
        batter_runs=("runs", "sum"),
        bowler_runs=("bowler_runs_conceded", "sum")
    )
)

print("\nLeg-bye deliveries with non-zero bowler runs:")
display(
    bowling_data[
        (bowling_data["extras_type"] == "legbyes") &
        (bowling_data["bowler_runs_conceded"] != 0)
    ][
        ["match_id", "bowler_name", "runs", "bowler_runs_conceded"]
    ]
)

,deliveries,batter_runs,bowler_runs
extras_type,,,
Overstep,268,398,694
byes,212,0,0
legbyes,1196,1,0
other,127,210,210
wide,3792,0,3792
NaN,89077,131419,131419



Leg-bye deliveries with non-zero bowler runs:


,match_id,bowler_name,runs,bowler_runs_conceded


In [28]:
other_data = bowling_data[
    bowling_data["extras_type"] == "other"
].copy()

print("Other extras summary:")
display(
    other_data.groupby(
        ["runs", "extras_2_runs"], dropna=False
    ).size().reset_index(name="deliveries")
)

print("\nOther deliveries with additional extras:")
display(
    other_data[other_data["extras_2_runs"].fillna(0) > 0][
        [
            "match_id",
            "innings",
            "overid",
            "bowler_name",
            "runs",
            "extras_2_runs",
            "is_wicket",
            "dismissal_type"
        ]
    ]
)

Other extras summary:


,runs,extras_2_runs,deliveries
0,0,0,55
1,0,1,4
2,0,4,4
3,1,0,23
4,2,0,4
5,3,0,1
6,4,0,20
7,6,0,16



Other deliveries with additional extras:


,match_id,innings,overid,bowler_name,runs,extras_2_runs,is_wicket,dismissal_type
8195,14127,2,9,Obed McCoy,0,4,False,NaN
15653,14159,2,9,Umran Malik,0,1,False,NaN
19721,20188,1,19,KM Asif,0,1,False,NaN
30929,20234,2,13,Harshal Patel,0,4,False,NaN
33081,20243,2,19,Naveen ul Haq,0,4,False,NaN
34950,21583,1,20,Mohammed Shami,0,1,False,NaN
54990,37093,2,19,Abdul Rahmani,0,1,False,NaN
56731,40695,1,18,Jofra Archer,0,4,False,NaN


In [29]:
# Rebuild recent IPL bowling data using corrected runs
recent_bowling_data = bowling_data[
    (bowling_data["comp_name"].str.startswith("IPL", na=False)) &
    (bowling_data["season"].isin(["2025", "2026"]))
].copy()

# Recalculate bowler statistics
recent_bowling_stats = (
    recent_bowling_data.groupby(["bowler_id", "bowler_name"])
    .agg(
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("legal_ball", "sum"),
        wickets=("bowler_wicket", "sum"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

# Calculate bowling metrics
recent_bowling_stats["overs"] = (
    recent_bowling_stats["legal_balls"] / 6
)

recent_bowling_stats["economy"] = (
    recent_bowling_stats["runs_conceded"] /
    recent_bowling_stats["overs"].replace(0, np.nan)
)

recent_bowling_stats["bowling_average"] = (
    recent_bowling_stats["runs_conceded"] /
    recent_bowling_stats["wickets"].replace(0, np.nan)
)

recent_bowling_stats["bowling_strike_rate"] = (
    recent_bowling_stats["legal_balls"] /
    recent_bowling_stats["wickets"].replace(0, np.nan)
)

recent_bowling_stats = recent_bowling_stats.round(2)

# Display top 25 by wickets
display(
    recent_bowling_stats.sort_values(
        ["wickets", "economy"],
        ascending=[False, True]
    ).head(25)
)

,bowler_id,bowler_name,runs_conceded,legal_balls,wickets,matches_played,overs,economy,bowling_average,bowling_strike_rate
3,360,Bhuvneshwar Kumar,983,690,45,30,115.00,8.55,21.84,15.33
60,4489,Prasidh Krishna,894,588,41,27,98.00,9.12,21.80,14.34
122,7967,Noor Ahmad,825,606,37,28,101.00,8.17,22.30,16.38
7,830,Josh Hazlewood,880,573,37,25,95.50,9.21,23.78,15.49
37,3298,Jofra Archer,973,634,36,28,105.67,9.21,27.03,17.61
77,5944,Mohammed Siraj,1080,714,35,32,119.00,9.08,30.86,20.40
92,6464,Arshdeep Singh,1050,669,35,30,111.50,9.42,30.00,19.11
157,39657,Eshan Malinga,745,486,33,22,81.00,9.20,22.58,14.73
51,4219,Krunal Pandya,802,579,31,30,96.50,8.31,25.87,18.68
40,3628,Kagiso Rabada,787,472,31,21,78.67,10.00,25.39,15.23


In [30]:

# Refresh recent IPL bowling data after the runs correction
recent_bowling_data = bowling_data[
    (bowling_data["comp_name"].str.startswith("IPL", na=False)) &
    (bowling_data["season"].isin(["2025", "2026"]))
].copy()

# Divide innings into bowling phases
recent_bowling_data["phase"] = pd.cut(
    recent_bowling_data["overid"],
    bins=[0, 6, 15, 20],
    labels=["Powerplay", "Middle", "Death"]
)

# Calculate phase-wise bowling performance
phase_stats = (
    recent_bowling_data.groupby(
        ["bowler_id", "bowler_name", "phase"],
        observed=True
    )
    .agg(
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("legal_ball", "sum"),
        wickets=("bowler_wicket", "sum"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

phase_stats = phase_stats[phase_stats["legal_balls"] >= 60].copy()

phase_stats["economy"] = (
    phase_stats["runs_conceded"] / (phase_stats["legal_balls"] / 6)
)

phase_stats["bowling_average"] = (
    phase_stats["runs_conceded"] /
    phase_stats["wickets"].replace(0, np.nan)
)

phase_stats["bowling_strike_rate"] = (
    phase_stats["legal_balls"] /
    phase_stats["wickets"].replace(0, np.nan)
)

phase_stats = phase_stats.round(2)

# Display top 10 wicket-takers in each phase
for phase in ["Powerplay", "Middle", "Death"]:
    print(f"\n{phase} Bowling - Top 10")
    display(
        phase_stats[phase_stats["phase"] == phase]
        .sort_values(
            ["wickets", "economy"],
            ascending=[False, True]
        )
        .head(10)
    )



Powerplay Bowling - Top 10


,bowler_id,bowler_name,phase,runs_conceded,legal_balls,wickets,matches_played,economy,bowling_average,bowling_strike_rate
9,360,Bhuvneshwar Kumar,Powerplay,484,390,25,30,7.45,19.36,15.60
195,5944,Mohammed Siraj,Powerplay,709,522,24,32,8.15,29.54,21.75
107,3628,Kagiso Rabada,Powerplay,530,312,21,21,10.19,25.24,14.86
98,3298,Jofra Archer,Powerplay,527,354,20,28,8.93,26.35,17.70
20,830,Josh Hazlewood,Powerplay,420,288,17,25,8.75,24.71,16.94
232,6464,Arshdeep Singh,Powerplay,531,360,15,30,8.85,35.40,24.00
35,1566,Mohammed Shami,Powerplay,451,313,14,22,8.65,32.21,22.36
46,1813,Deepak Chahar,Powerplay,436,300,14,22,8.72,31.14,21.43
168,4816,Khaleel Ahmed,Powerplay,429,300,12,19,8.58,35.75,25.00
367,27964,Anshul Kamboj,Powerplay,387,252,12,22,9.21,32.25,21.00



Middle Bowling - Top 10


,bowler_id,bowler_name,phase,runs_conceded,legal_balls,wickets,matches_played,economy,bowling_average,bowling_strike_rate
307,7967,Noor Ahmad,Middle,640,480,28,28,8.00,22.86,17.14
164,4722,Rashid Khan,Middle,767,527,25,32,8.73,30.68,21.08
132,4219,Krunal Pandya,Middle,649,492,24,29,7.91,27.04,20.50
155,4489,Prasidh Krishna,Middle,443,330,20,26,8.05,22.15,16.50
115,3644,Kuldeep Yadav,Middle,628,444,20,25,8.49,31.40,22.20
41,1612,Yuzvendra Chahal,Middle,695,450,20,24,9.27,34.75,22.50
205,6256,Varun Chakravarthy,Middle,522,402,19,24,7.79,27.47,21.16
4,152,Ravindra Jadeja,Middle,567,403,19,26,8.44,29.84,21.21
186,5577,Sai Kishore,Middle,348,222,15,15,9.41,23.20,14.80
15,781,Sunil Narine,Middle,518,420,14,25,7.40,37.00,30.00



Death Bowling - Top 10


,bowler_id,bowler_name,phase,runs_conceded,legal_balls,wickets,matches_played,economy,bowling_average,bowling_strike_rate
11,360,Bhuvneshwar Kumar,Death,415,252,18,25,9.88,23.06,14.00
156,4489,Prasidh Krishna,Death,303,168,18,21,10.82,16.83,9.33
396,39657,Eshan Malinga,Death,260,156,15,19,10.00,17.33,10.40
234,6464,Arshdeep Singh,Death,336,195,15,25,10.34,22.40,13.00
19,829,Mitchell Starc,Death,146,90,14,14,9.73,10.43,6.43
335,8746,Vaibhav Arora,Death,266,153,14,19,10.43,19.00,10.93
369,27964,Anshul Kamboj,Death,238,131,14,13,10.90,17.00,9.36
56,2320,Trent Boult,Death,185,119,12,20,9.33,15.42,9.92
22,830,Josh Hazlewood,Death,283,164,12,20,10.35,23.58,13.67
301,7808,Marco Jansen,Death,250,135,12,20,11.11,20.83,11.25


In [31]:

# 1. Overall pace vs spin comparison
style_summary = (
    recent_bowling_data.groupby("pace_or_spin")
    .agg(
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("legal_ball", "sum"),
        wickets=("bowler_wicket", "sum"),
        deliveries=("bowler_name", "size"),
        bowlers=("bowler_id", "nunique")
    )
    .reset_index()
)

style_summary["economy"] = (
    style_summary["runs_conceded"] /
    (style_summary["legal_balls"] / 6)
)

style_summary["bowling_average"] = (
    style_summary["runs_conceded"] /
    style_summary["wickets"].replace(0, np.nan)
)

style_summary["bowling_strike_rate"] = (
    style_summary["legal_balls"] /
    style_summary["wickets"].replace(0, np.nan)
)

print("PACE VS SPIN COMPARISON")
display(style_summary.round(2))


# 2. Individual bowler performance by style
style_bowling_stats = (
    recent_bowling_data.groupby(
        ["bowler_id", "bowler_name", "pace_or_spin"]
    )
    .agg(
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("legal_ball", "sum"),
        wickets=("bowler_wicket", "sum"),
        matches_played=("match_id", "nunique")
    )
    .reset_index()
)

# Require at least 60 legal balls
style_bowling_stats = style_bowling_stats[
    style_bowling_stats["legal_balls"] >= 60
].copy()

style_bowling_stats["economy"] = (
    style_bowling_stats["runs_conceded"] /
    (style_bowling_stats["legal_balls"] / 6)
)

style_bowling_stats["bowling_average"] = (
    style_bowling_stats["runs_conceded"] /
    style_bowling_stats["wickets"].replace(0, np.nan)
)

style_bowling_stats["bowling_strike_rate"] = (
    style_bowling_stats["legal_balls"] /
    style_bowling_stats["wickets"].replace(0, np.nan)
)

style_bowling_stats = style_bowling_stats.round(2)

# 3. Top 15 pace and spin bowlers
for style in ["Pace", "Spin"]:
    print(f"\nTOP 15 {style.upper()} BOWLERS")

    display(
        style_bowling_stats[
            style_bowling_stats["pace_or_spin"] == style
        ]
        .sort_values(
            ["wickets", "economy"],
            ascending=[False, True]
        )
        .head(15)
    )


PACE VS SPIN COMPARISON


,pace_or_spin,runs_conceded,legal_balls,wickets,deliveries,bowlers,economy,bowling_average,bowling_strike_rate
0,Pace,35974,21867,1131,23107,106,9.87,31.81,19.33
1,Spin,17077,11360,536,11705,64,9.02,31.86,21.19



TOP 15 PACE BOWLERS


,bowler_id,bowler_name,pace_or_spin,runs_conceded,legal_balls,wickets,matches_played,economy,bowling_average,bowling_strike_rate
3,360,Bhuvneshwar Kumar,Pace,983,690,45,30,8.55,21.84,15.33
60,4489,Prasidh Krishna,Pace,894,588,41,27,9.12,21.80,14.34
7,830,Josh Hazlewood,Pace,880,573,37,25,9.21,23.78,15.49
37,3298,Jofra Archer,Pace,973,634,36,28,9.21,27.03,17.61
77,5944,Mohammed Siraj,Pace,1080,714,35,32,9.08,30.86,20.40
92,6464,Arshdeep Singh,Pace,1050,669,35,30,9.42,30.00,19.11
157,39657,Eshan Malinga,Pace,745,486,33,22,9.20,22.58,14.73
40,3628,Kagiso Rabada,Pace,787,472,31,21,10.00,25.39,15.23
145,27964,Anshul Kamboj,Pace,702,431,29,22,9.77,24.21,14.86
132,8746,Vaibhav Arora,Pace,820,483,28,22,10.19,29.29,17.25



TOP 15 SPIN BOWLERS


,bowler_id,bowler_name,pace_or_spin,runs_conceded,legal_balls,wickets,matches_played,economy,bowling_average,bowling_strike_rate
122,7967,Noor Ahmad,Spin,825,606,37,28,8.17,22.30,16.38
51,4219,Krunal Pandya,Spin,802,579,31,30,8.31,25.87,18.68
63,4722,Rashid Khan,Spin,1022,671,30,32,9.14,34.07,22.37
5,781,Sunil Narine,Spin,691,579,29,25,7.16,23.83,19.97
81,6256,Varun Chakravarthy,Spin,743,546,28,24,8.16,26.54,19.50
15,1612,Yuzvendra Chahal,Spin,810,516,28,25,9.42,28.93,18.43
43,3644,Kuldeep Yadav,Spin,770,540,25,25,8.56,30.80,21.60
1,152,Ravindra Jadeja,Spin,601,425,20,26,8.48,30.05,21.25
73,5577,Sai Kishore,Spin,439,279,20,17,9.44,21.95,13.95
114,7503,Ravi Bishnoi,Spin,658,378,20,20,10.44,32.90,18.90


In [32]:
# Start with batter runs
bowling_data["bowler_runs_conceded"] = bowling_data["runs"].copy()

# Byes and leg-byes are not charged to the bowler
bowling_data.loc[
    bowling_data["extras_type"].isin(["byes", "legbyes"]),
    "bowler_runs_conceded"
] = 0

# Add one penalty run for wides
bowling_data.loc[
    bowling_data["extras_type"] == "wide",
    "bowler_runs_conceded"
] += 1

# Add one penalty run and additional extras for no-balls
bowling_data.loc[
    bowling_data["extras_type"] == "Overstep",
    "bowler_runs_conceded"
] += (
    1 + bowling_data.loc[
        bowling_data["extras_type"] == "Overstep",
        "extras_2_runs"
    ].fillna(0)
)

# Keep the additional 20 runs under "other" uncharged
# until their scoring classification is verified

# Audit
display(
    bowling_data.groupby("extras_type", dropna=False)
    .agg(
        deliveries=("bowler_name", "size"),
        bowler_runs=("bowler_runs_conceded", "sum")
    )
)

,deliveries,bowler_runs
extras_type,,
Overstep,268,694
byes,212,0
legbyes,1196,0
other,127,210
wide,3792,3792
NaN,89077,131419


In [33]:
# Recreate recent IPL dataset using corrected bowling runs
recent_bowling_data = bowling_data[
    (bowling_data["comp_name"].str.startswith("IPL", na=False)) &
    (bowling_data["season"].isin(["2025", "2026"]))
].copy()

# Ensure legal-ball column exists
recent_bowling_data["legal_ball"] = (
    ~recent_bowling_data["extras_type"].isin(["wide", "Overstep"])
)

print("Recent IPL deliveries:", len(recent_bowling_data))
print("Recent IPL bowlers:", recent_bowling_data["bowler_id"].nunique())

Recent IPL deliveries: 34812
Recent IPL bowlers: 170


In [34]:
# Check the actual bowling runs in the source dataset
print(
    bowling_data.groupby("extras_type", dropna=False)
    .agg(
        deliveries=("bowler_runs_conceded", "size"),
        bowler_runs=("bowler_runs_conceded", "sum")
    )
)

# Confirm no leg-bye runs are charged to bowlers
print(
    "Leg-bye runs charged:",
    bowling_data.loc[
        bowling_data["extras_type"] == "legbyes",
        "bowler_runs_conceded"
    ].sum()
)

# Confirm other extras are not including the uncertain additional extras
print(
    "Other extras runs charged:",
    bowling_data.loc[
        bowling_data["extras_type"] == "other",
        "bowler_runs_conceded"
    ].sum()
)

             deliveries  bowler_runs
extras_type                         
Overstep            268          694
byes                212            0
legbyes            1196            0
other               127          210
wide               3792         3792
NaN               89077       131419
Leg-bye runs charged: 0
Other extras runs charged: 210


In [35]:
recent_bowling_data = bowling_data[
    (bowling_data["comp_name"].str.startswith("IPL", na=False)) &
    (bowling_data["season"].isin(["2025", "2026"]))
].copy()

pace_spin_stats = (
    recent_bowling_data.groupby("pace_or_spin")
    .agg(
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("legal_ball", "sum"),
        wickets=("bowler_wicket", "sum"),
        deliveries=("bowler_id", "count"),
        bowlers=("bowler_id", "nunique")
    )
    .reset_index()
)

pace_spin_stats["economy"] = (
    pace_spin_stats["runs_conceded"] /
    (pace_spin_stats["legal_balls"] / 6)
)

pace_spin_stats["bowling_average"] = (
    pace_spin_stats["runs_conceded"] /
    pace_spin_stats["wickets"]
)

pace_spin_stats["bowling_strike_rate"] = (
    pace_spin_stats["legal_balls"] /
    pace_spin_stats["wickets"]
)

print(pace_spin_stats.round(2))

  pace_or_spin  runs_conceded  legal_balls  wickets  deliveries  bowlers  \
0         Pace          35974        21867     1131       23107      106   
1         Spin          17077        11360      536       11705       64   

   economy  bowling_average  bowling_strike_rate  
0     9.87            31.81                19.33  
1     9.02            31.86                21.19  


In [36]:
pace_spin_stats = (
    bowling_data.groupby("pace_or_spin")
    .agg(
        runs_conceded=("bowler_runs_conceded", "sum"),
        legal_balls=("legal_ball", "sum"),
        wickets=("bowler_wicket", "sum"),
        deliveries=("bowler_id", "count"),
        bowlers=("bowler_id", "nunique")
    )
    .reset_index()
)

pace_spin_stats["economy"] = (
    pace_spin_stats["runs_conceded"] /
    (pace_spin_stats["legal_balls"] / 6)
)

pace_spin_stats["bowling_average"] = (
    pace_spin_stats["runs_conceded"] /
    pace_spin_stats["wickets"]
)

pace_spin_stats["bowling_strike_rate"] = (
    pace_spin_stats["legal_balls"] /
    pace_spin_stats["wickets"]
)

print(pace_spin_stats.round(2))

  pace_or_spin  runs_conceded  legal_balls  wickets  deliveries  bowlers  \
0         Pace          89636        57181     3013       60365      203   
1         Spin          46479        33431     1557       34307      149   

   economy  bowling_average  bowling_strike_rate  
0     9.41            29.75                18.98  
1     8.34            29.85                21.47  
